In [10]:
#!/usr/bin/env python3

This file illustrates how you might experiment with the HMM interface.
You can paste these commands in at the Python prompt, or execute `test_en.py` directly.
A notebook interface is nicer than the plain Python prompt, so we provide
a notebook version of this file as `test_en.ipynb`, which you can open with
`jupyter` or with Visual Studio `code` (run it with the `nlp-class` kernel).

In [11]:
import logging
import math
import os
from pathlib import Path

In [12]:
from corpus import TaggedCorpus
from eval import eval_tagging, model_cross_entropy, viterbi_error_rate
from hmm import HiddenMarkovModel
from crf import ConditionalRandomField

Set up logging.

In [13]:
logging.root.setLevel(level=logging.INFO)
log = logging.getLogger("test_en")       # For usage, see findsim.py in earlier assignment.
logging.basicConfig(format="%(levelname)s : %(message)s", level=logging.INFO)  # could change INFO to DEBUG

Switch working directory to the directory where the data live.  You may need to edit this line.

In [14]:
os.chdir("../data")

In [15]:
entrain = TaggedCorpus(Path("ensup"), Path("enraw"))                               # all training
ensup =   TaggedCorpus(Path("ensup"), tagset=entrain.tagset, vocab=entrain.vocab)  # supervised training
endev =   TaggedCorpus(Path("endev"), tagset=entrain.tagset, vocab=entrain.vocab)  # evaluation
print(f"{len(entrain)=}  {len(ensup)=}  {len(endev)=}")

INFO : Read 191873 tokens from ensup, enraw
INFO : Created 26 tag types
INFO : Created 18461 word types


len(entrain)=8064  len(ensup)=4051  len(endev)=996


In [16]:
known_vocab = TaggedCorpus(Path("ensup")).vocab    # words seen with supervised tags; used in evaluation
log.info(f"Tagset: f{list(entrain.tagset)}")

INFO : Read 95936 tokens from ensup
INFO : Created 26 tag types
INFO : Created 12466 word types
INFO : Tagset: f['W', 'J', 'N', 'C', 'V', 'I', 'D', ',', 'M', 'P', '.', 'E', 'R', '`', "'", 'T', '$', ':', '-', '#', 'S', 'F', 'U', 'L', '_EOS_TAG_', '_BOS_TAG_']


Make an HMM.  Let's do some pre-training to approximately maximize the
regularized log-likelihood on supervised training data.  In other words, the
probabilities at the M step will just be supervised count ratios.

On each epoch, you will see two progress bars: first it collects counts from
all the sentences (E step), and then after the M step, it evaluates the loss
function, which is the (unregularized) cross-entropy on the training set.

The parameters don't actually matter during the E step because there are no
hidden tags to impute.  The first M step will jump right to the optimal
solution.  The code will try a second epoch with the revised parameters, but
the result will be identical, so it will detect convergence and stop.

We arbitrarily choose λ=1 for our add-λ smoothing at the M step, but it would
be better to search for the best value of this hyperparameter.

In [17]:
log.info("*** Hidden Markov Model (HMM)")
hmm = HiddenMarkovModel(entrain.tagset, entrain.vocab)  # randomly initialized parameters  
loss_sup = lambda model: model_cross_entropy(model, eval_corpus=ensup)
hmm.train(corpus=ensup, loss=loss_sup, λ=1.0,
          save_path="ensup_hmm.pkl") 

INFO : *** Hidden Markov Model (HMM)
100%|██████████| 4051/4051 [00:03<00:00, 1203.65it/s]
INFO : Cross-entropy: 12.6439 nats (= perplexity 309875.493)
100%|██████████| 4051/4051 [00:03<00:00, 1196.10it/s]
INFO : Cross-entropy: 7.4503 nats (= perplexity 1720.390)
100%|██████████| 4051/4051 [00:03<00:00, 1195.42it/s]
INFO : Cross-entropy: 7.4503 nats (= perplexity 1720.399)
100%|██████████| 4051/4051 [00:03<00:00, 1183.58it/s]
INFO : Cross-entropy: 7.4503 nats (= perplexity 1720.393)
100%|██████████| 4051/4051 [00:03<00:00, 1176.71it/s]
INFO : Cross-entropy: 7.4503 nats (= perplexity 1720.401)
100%|██████████| 4051/4051 [00:03<00:00, 1153.59it/s]
INFO : Cross-entropy: 7.4503 nats (= perplexity 1720.403)
100%|██████████| 4051/4051 [00:03<00:00, 1171.67it/s]
INFO : Cross-entropy: 7.4503 nats (= perplexity 1720.403)
100%|██████████| 4051/4051 [00:03<00:00, 1188.31it/s]
INFO : Cross-entropy: 7.4503 nats (= perplexity 1720.403)
100%|██████████| 4051/4051 [00:03<00:00, 1184.15it/s]
INFO : Cro

Now let's throw in the unsupervised training data as well, and continue
training as before, in order to increase the regularized log-likelihood on
this larger, semi-supervised training set.  It's now the *incomplete-data*
log-likelihood.

This time, we'll use a different evaluation loss function: we'll stop when the
*tagging error rate* on a held-out dev set stops getting better.  Also, the
implementation of this loss function (`viterbi_error_rate`) includes a helpful
side effect: it logs the *cross-entropy* on the held-out dataset as well, just
for your information.

We hope that held-out tagging accuracy will go up for a little bit before it
goes down again (see Merialdo 1994). (Log-likelihood on training data will
continue to improve, and that improvement may generalize to held-out
cross-entropy.  But getting accuracy to increase is harder.)

In [18]:
hmm = HiddenMarkovModel.load("ensup_hmm.pkl")  # reset to supervised model (in case you're re-executing this bit)
loss_dev = lambda model: viterbi_error_rate(model, eval_corpus=endev, 
                                            known_vocab=known_vocab)
hmm.train(corpus=entrain, loss=loss_dev, λ=1.0,
          save_path="entrain_hmm.pkl")

INFO : Loaded model from ensup_hmm.pkl
100%|██████████| 996/996 [00:01<00:00, 843.99it/s]
INFO : Cross-entropy: 7.5993 nats (= perplexity 1996.798)
100%|██████████| 996/996 [00:01<00:00, 538.58it/s]
INFO : Tagging accuracy: all: 88.663%, known: 93.059%, seen: 44.108%, novel: 42.734%
100%|██████████| 996/996 [00:01<00:00, 880.36it/s]
INFO : Cross-entropy: 7.3485 nats (= perplexity 1553.846)
100%|██████████| 996/996 [00:01<00:00, 589.78it/s]
INFO : Tagging accuracy: all: 87.031%, known: 91.397%, seen: 45.791%, novel: 40.225%
100%|██████████| 996/996 [00:01<00:00, 878.68it/s]
INFO : Cross-entropy: 7.3543 nats (= perplexity 1562.953)
100%|██████████| 996/996 [00:01<00:00, 589.40it/s]
INFO : Tagging accuracy: all: 85.887%, known: 90.174%, seen: 45.960%, novel: 39.696%
100%|██████████| 996/996 [00:01<00:00, 880.09it/s]
INFO : Cross-entropy: 7.3607 nats (= perplexity 1572.952)
100%|██████████| 996/996 [00:01<00:00, 589.34it/s]
INFO : Tagging accuracy: all: 85.219%, known: 89.446%, seen: 45.28

You can also retry the above workflow where you start with a worse supervised
model (like Merialdo).  Does EM help more in that case?  It's easiest to rerun
exactly the code above, but first make the `ensup` file smaller by copying
`ensup-tiny` over it.  `ensup-tiny` is only 25 sentences (that happen to cover
all tags in `endev`).  Back up your old `ensup` and your old `*.pkl` models
before you do this.

More detailed look at the first 10 sentences in the held-out corpus,
including Viterbi tagging.

In [19]:
def look_at_your_data(model, dev, N):
    for m, sentence in enumerate(dev):
        if m >= N: break
        viterbi = model.viterbi_tagging(sentence.desupervise(), endev)
        counts = eval_tagging(predicted=viterbi, gold=sentence, 
                              known_vocab=known_vocab)
        num = counts['NUM', 'ALL']
        denom = counts['DENOM', 'ALL']
        
        log.info(f"Gold:    {sentence}")
        log.info(f"Viterbi: {viterbi}")
        log.info(f"Loss:    {denom - num}/{denom}")
        xent = -model.logprob(sentence, endev) / len(sentence)  # measured in nats
        log.info(f"Cross-entropy: {xent/math.log(2)} nats (= perplexity {math.exp(xent)})\n---")

In [20]:
look_at_your_data(hmm, endev, 10)

INFO : Gold:    ``/` We/P 're/V strongly/R _OOV_/V that/I anyone/N who/W has/V eaten/V in/I the/D cafeteria/N this/D month/N have/V the/D shot/N ,/, ''/' Mr./N Mattausch/N added/V ,/, ``/` and/C that/D means/V virtually/R everyone/N who/W works/V here/R ./.
INFO : Viterbi: ``/` We/P 're/V strongly/R _OOV_/V that/I anyone/N who/W has/V eaten/V in/I the/D cafeteria/N this/D month/N have/V the/D shot/N ,/, ''/' Mr./N Mattausch/I added/N ,/, ``/N and/C that/I means/V virtually/R everyone/, who/W works/V here/R ./.
INFO : Loss:    5/34
INFO : Cross-entropy: 10.661638259887695 nats (= perplexity 1619.8426059130436)
---
INFO : Gold:    I/P was/V _OOV_/V to/T read/V the/D _OOV_/N of/I facts/N in/I your/P Oct./N 13/C editorial/N ``/` _OOV_/N 's/P _OOV_/N _OOV_/N ./. ''/'
INFO : Viterbi: I/P was/V _OOV_/V to/T read/V the/D _OOV_/N of/I facts/N in/I your/D Oct./N 13/C editorial/J ``/N _OOV_/I 's/P _OOV_/J _OOV_/N ./. ''/'
INFO : Loss:    5/21
INFO : Cross-entropy: 10.964547157287598 nats (= perpl

Now let's try supervised training of a CRF (this doesn't use the unsupervised
part of the data, so it is comparable to the supervised pre-training we did
for the HMM).  We will use SGD to approximately maximize the regularized
log-likelihood. 

As with the semi-supervised HMM training, we'll periodically evaluate the
tagging accuracy (and also print the cross-entropy) on a held-out dev set.
We use the default `eval_interval` and `tolerance`.  If you want to stop
sooner, then you could increase the `tolerance` so the training method decides
sooner that it has converged.

We arbitrarily choose reg = 1.0 for L2 regularization, learning rate = 0.05,
and a minibatch size of 10, but it would be better to search for the best
value of these hyperparameters.

Note that the logger reports the CRF's *conditional* cross-entropy, log p(tags
| words) / n.  This is much lower than the HMM's *joint* cross-entropy log
p(tags, words) / n, but that doesn't mean the CRF is worse at tagging.  The
CRF is just predicting less information.

In [21]:
log.info("*** Conditional Random Field (CRF)\n")
crf = ConditionalRandomField(entrain.tagset, entrain.vocab)  # randomly initialized parameters  
crf.train(corpus=ensup, loss=loss_dev, reg=1.0, lr=0.05, minibatch_size=10,
          save_path="ensup_crf.pkl")

INFO : *** Conditional Random Field (CRF)

100%|██████████| 996/996 [00:01<00:00, 654.20it/s]
INFO : Cross-entropy: 3.0539 nats (= perplexity 21.198)
100%|██████████| 996/996 [00:01<00:00, 720.77it/s]
INFO : Tagging accuracy: all: 2.447%, known: 2.450%, seen: 3.367%, novel: 2.048%
100%|██████████| 996/996 [00:01<00:00, 655.46it/s]
INFO : Cross-entropy: 0.9119 nats (= perplexity 2.489)
100%|██████████| 996/996 [00:01<00:00, 712.97it/s]
INFO : Tagging accuracy: all: 72.441%, known: 73.518%, seen: 57.912%, novel: 62.616%
100%|██████████| 996/996 [00:01<00:00, 625.29it/s]
INFO : Cross-entropy: 0.7519 nats (= perplexity 2.121)
100%|██████████| 996/996 [00:01<00:00, 691.65it/s]
INFO : Tagging accuracy: all: 75.218%, known: 76.984%, seen: 55.556%, novel: 57.464%
100%|██████████| 996/996 [00:01<00:00, 643.57it/s]
INFO : Cross-entropy: 0.6584 nats (= perplexity 1.932)
100%|██████████| 996/996 [00:01<00:00, 718.90it/s]
INFO : Tagging accuracy: all: 78.730%, known: 80.285%, seen: 61.785%, novel: 

Let's examine how the CRF does on individual sentences. 
(Do you see any error patterns here that would inspire additional CRF features?)

In [22]:
look_at_your_data(crf, endev, 10)

INFO : Gold:    ``/` We/P 're/V strongly/R _OOV_/V that/I anyone/N who/W has/V eaten/V in/I the/D cafeteria/N this/D month/N have/V the/D shot/N ,/, ''/' Mr./N Mattausch/N added/V ,/, ``/` and/C that/D means/V virtually/R everyone/N who/W works/V here/R ./.
INFO : Viterbi: ``/` We/P 're/V strongly/N _OOV_/N that/I anyone/N who/W has/V eaten/N in/I the/D cafeteria/N this/D month/N have/V the/D shot/N ,/, ''/' Mr./N Mattausch/N added/V ,/, ``/` and/C that/I means/N virtually/N everyone/N who/W works/V here/R ./.
INFO : Loss:    6/34
INFO : Cross-entropy: 0.49836137890815735 nats (= perplexity 1.4126082170207672)
---
INFO : Gold:    I/P was/V _OOV_/V to/T read/V the/D _OOV_/N of/I facts/N in/I your/P Oct./N 13/C editorial/N ``/` _OOV_/N 's/P _OOV_/N _OOV_/N ./. ''/'
INFO : Viterbi: I/P was/V _OOV_/V to/T read/V the/D _OOV_/N of/I facts/N in/I your/P Oct./N 13/C editorial/N ``/` _OOV_/N 's/P _OOV_/N _OOV_/N ./. ''/'
INFO : Loss:    0/21
INFO : Cross-entropy: 0.26305127143859863 nats (= per